# Task 6: K-Nearest Neighbors (KNN) Classification

**AI & ML Internship — Elevate Labs**

---

## Objective
Implement and understand KNN for classification using the Iris dataset. Covers distance-based instance learning, feature normalization, hyperparameter tuning for K, model evaluation, and decision boundary visualization.

**Key goals:**
1. Load and explore the Iris classification dataset
2. Normalize features for distance-based algorithm
3. Implement KNN with KNeighborsClassifier
4. Experiment with different values of K
5. Evaluate model using accuracy, confusion matrix, classification report
6. Visualize decision boundaries
7. Prepare for common KNN interview questions

## Dataset Description

**Dataset:** Iris Dataset
**Source:** `dataset/iris.csv`
**Samples:** 150 rows × 4 features + target

### Features
| Feature | Description |
|---------|-------------|
| sepal_length | Sepal length in cm |
| sepal_width | Sepal width in cm |
| petal_length | Petal length in cm |
| petal_width | Petal width in cm |

### Target
| Class | Value |
|-------|-------|
| Iris-setosa | 0 |
| Iris-versicolor | 1 |
| Iris-virginica | 2 |

**Why this dataset?** Multi-class problem, clean data, well-known benchmark — ideal for understanding KNN fundamentals.

## Import Libraries

- `pandas` — data manipulation
- `numpy` — numerical operations
- `matplotlib.pyplot` — plotting
- `seaborn` — statistical visualizations
- `sklearn` — preprocessing, model training, evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully.")

## Load and Explore Dataset

Load the Iris dataset and inspect its structure.

In [ ]:
import os

dataset_path = "../dataset/iris.csv"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
display(df.head())

# Map target to numeric if needed
if df['species'].dtype == 'object':
    df['target'] = df['species'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2})
else:
    df['target'] = df['species']

print(f"\nClass distribution:")
print(df['species'].value_counts())

## Data Preprocessing

### Split Features and Target

Separate the feature columns from the target variable.

In [ ]:
X = df.drop(['species', 'target'], axis=1)
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

### Train-Test Split

Split data into training (80%) and testing (20%) sets with stratification to maintain class distribution.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts().sort_index())
print(f"\nClass distribution in testing set:")
print(y_test.value_counts().sort_index())

### Feature Normalization

KNN is a distance-based algorithm, so feature scaling is crucial. Use StandardScaler to normalize features to have mean=0 and variance=1.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalized using StandardScaler.")
print(f"\nTraining set mean: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Training set std: {X_train_scaled.std(axis=0).round(4)}")

## Baseline KNN Model (K=3)

Train a baseline KNN model with K=3 neighbors using Euclidean distance (default).

In [ ]:
knn_baseline = KNeighborsClassifier(n_neighbors=3)
knn_baseline.fit(X_train_scaled, y_train)

y_pred_baseline = knn_baseline.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline KNN (K=3) Accuracy: {baseline_accuracy:.4f}")
print(f"\nClassification Report (Baseline):")
print(classification_report(y_test, y_pred_baseline, target_names=['setosa', 'versicolor', 'virginica']))

## Experiment with Different Values of K

Test K values from 1 to 15 and plot accuracy to find the optimal K.

In [ ]:
k_values = range(1, 16)
train_accuracies = []
test_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_acc = knn.score(X_train_scaled, y_train)
    test_acc = knn.score(X_test_scaled, y_test)
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

plt.figure(figsize=(10, 6))
plt.plot(k_values, train_accuracies, 'o-', label='Training Accuracy', color='blue')
plt.plot(k_values, test_accuracies, 's-', label='Testing Accuracy', color='orange')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN: Accuracy vs. K Value')
plt.legend()
plt.grid(True)
plt.xticks(k_values)
plt.show()

best_k = k_values[np.argmax(test_accuracies)]
best_accuracy = max(test_accuracies)
print(f"\nBest K: {best_k}")
print(f"Best Test Accuracy: {best_accuracy:.4f}")

## Model Evaluation with Optimal K

Evaluate the best model using confusion matrix and detailed classification report.

In [ ]:
knn_optimal = KNeighborsClassifier(n_neighbors=best_k)
knn_optimal.fit(X_train_scaled, y_train)
y_pred_optimal = knn_optimal.predict(X_test_scaled)

print(f"=== Optimal KNN (K={best_k}) Results ===")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_optimal):.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_optimal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['setosa', 'versicolor', 'virginica'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', ax=ax)
plt.title(f'Confusion Matrix (K={best_k})')
plt.show()

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_optimal, target_names=['setosa', 'versicolor', 'virginica']))

## Decision Boundary Visualization

Visualize how KNN creates decision boundaries using two features at a time.

In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_boundary(X, y, model, feature_idx, feature_names):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 7))
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
    
    plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.8)
    
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
                         edgecolor='black', s=50)
    
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title(f'KNN Decision Boundaries (K={best_k}) - {feature_names[0]} vs {feature_names[1]}')
    plt.legend(handles=scatter.legend_elements()[0], 
               labels=['setosa', 'versicolor', 'virginica'])
    plt.show()

feature_pairs = [
    (0, 1, 'sepal length (cm)', 'sepal width (cm)'),
    (2, 3, 'petal length (cm)', 'petal width (cm)'),
]

for idx1, idx2, name1, name2 in feature_pairs:
    X_pair = X_train_scaled[:, [idx1, idx2]]
    knn_pair = KNeighborsClassifier(n_neighbors=best_k)
    knn_pair.fit(X_pair, y_train)
    plot_decision_boundary(X_pair, y_train, knn_pair, 
                           [name1, name2], [name1, name2])

## Interview Questions & Answers

### 1. How does the KNN algorithm work?

KNN is an instance-based learning algorithm. For a new data point:
1. Calculate the distance between the new point and all training points
2. Select the K nearest neighbors
3. Take a majority vote among the K neighbors for classification
4. Assign the class with the most votes to the new point

### 2. How do you choose the right K?

- Use cross-validation and plot accuracy vs. K
- Small K: low bias, high variance (overfitting)
- Large K: high bias, low variance (underfitting)
- Rule of thumb: K = sqrt(N) where N is number of samples, or use odd K for binary classification to avoid ties

### 3. Why is normalization important in KNN?

KNN uses distance metrics (Euclidean, Manhattan, etc.) to find neighbors. If features have different scales, features with larger scales dominate the distance calculation, leading to biased results. Normalization ensures all features contribute equally.

### 4. What is the time complexity of KNN?

- **Training:** O(1) — KNN is a lazy learner, it just stores the data
- **Prediction:** O(N*D) — for each prediction, it calculates distance to all N training samples with D features
- For KNN with K neighbors: O(N*D) per prediction

### 5. What are pros and cons of KNN?

**Pros:**
- Simple and intuitive
- No training phase (lazy learning)
- Naturally handles multi-class problems
- Can be used for both classification and regression

**Cons:**
- Slow prediction on large datasets
- Sensitive to irrelevant features and noise
- Requires feature scaling
- Curse of dimensionality — performance degrades with many features

### 6. Is KNN sensitive to noise?

Yes, KNN is sensitive to noise. Noisy data points can be selected as neighbors, especially with small K. Using a larger K or weighted KNN (where closer neighbors have more weight) can reduce this sensitivity.

### 7. How does KNN handle multi-class problems?

KNN naturally handles multi-class problems by taking a majority vote among the K nearest neighbors. The class with the most votes is assigned. No modification is needed — it works the same way as binary classification.

### 8. What's the role of distance metrics in KNN?

Distance metrics define how similarity is measured between points:
- **Euclidean:** Straight-line distance (most common, default in sklearn)
- **Manhattan:** Sum of absolute differences (better for high dimensions)
- **Minkowski:** Generalized form of Euclidean and Manhattan
- **Cosine:** Angle between vectors (good for text/high-dimensional data)

The choice of distance metric significantly affects KNN performance and should be tuned based on the data characteristics.

In [ ]:
print("=== Summary ===")
print(f"\nDataset: Iris (150 samples, 4 features, 3 classes)")
print(f"Best K: {best_k}")
print(f"Best Test Accuracy: {best_accuracy:.4f}")
print(f"\nKey Takeaways:")
print("- KNN is simple but effective for small to medium datasets")
print("- Feature normalization is essential for distance-based algorithms")
print("- K selection significantly impacts model performance")
print("- Decision boundaries show how KNN partitions the feature space")
print("- KNN handles multi-class classification naturally")